# PC population network demo

This notebook builds two small homogeneous Purkinje-cell populations using `examples.neuron_compare.cell.pc_ma2024.pc_braincell.PC` and connects them with the `braincell.network` API.

The presynaptic PC population receives staggered soma current clamps. `Network.add_edges(...)` samples a probabilistic cell-level graph, and `Network.add_projection(...)` maps those edges to the placed postsynaptic `ExpSyn` pool.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import sys
from pathlib import Path


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import brainstate
import brainunit as u
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import mech
from braincell.filter import at

from examples.neuron_compare.cell.pc_ma2024.parameters import (
    DEFAULT_MORPH_PATH,
    load_pc24_params,
)
from examples.neuron_compare.cell.pc_ma2024.pc_braincell import PC as BrainCellPC

brainstate.environ.set(precision=64)

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)
print("morphology:", DEFAULT_MORPH_PATH)

In [ ]:
params = load_pc24_params()
temperature_celsius = 36.0
v_init_mV = -65.0


def pulse_protocol(size: int) -> mech.CurrentClamp:
    base_delays = np.asarray([0.2, 0.3, 0.4, 0.5], dtype=float)
    input_delays = base_delays[np.arange(size) % len(base_delays)]
    amplitudes = u.Quantity(1.8 + 0.05 * np.arange(size, dtype=float), u.nA)
    return mech.CurrentClamp(
        delay=u.Quantity(input_delays, u.ms),
        durations=0.5 * u.ms,
        amplitudes=amplitudes,
    )


def build_pre_pc(size: int = 4):
    pc = BrainCellPC(
        DEFAULT_MORPH_PATH,
        params=params,
        temperature_celsius=temperature_celsius,
        v_init_mV=v_init_mV,
        pop_size=(size,),
        name="pre_pc",
    ).build()
    cell = pc.cell
    cell.V_th = 0.0 * u.mV
    cell.place(at("soma", 0.5), pulse_protocol(size))
    cell.place(at("soma", 0.5), mech.StateProbe(name="v", field="v"))
    return cell


def build_post_pc(size: int = 4):
    pc = BrainCellPC(
        DEFAULT_MORPH_PATH,
        params=params,
        temperature_celsius=temperature_celsius,
        v_init_mV=v_init_mV,
        pop_size=(size,),
        name="post_pc",
    ).build()
    cell = pc.cell
    cell.V_th = 0.0 * u.mV
    cell.place(at("soma", 0.5), mech.StateProbe(name="v", field="v"))
    cell.place(at("soma", 0.5), mech.MechanismProbe(name="g_syn", mechanism="exp", field="g"))
    cell.place(at("soma", 0.5), mech.CurrentProbe(name="i_syn", mechanism="exp"))
    cell.place(
        at("soma", 0.5),
        mech.Synapse(
            "ExpSyn",
            tau=2.0 * u.ms,
            e=0.0 * u.mV,
            weight=1.0 * u.uS,
            name="exp",
        ),
    )
    return cell


def initialize_to_steady_state(*cells) -> None:
    for cell in cells:
        cell.init_state()
        cell.reset_state()

In [ ]:
n_pre = 2
n_post = 2
connection_probability = 0.5
connection_seed = 7
synaptic_weight = 0.005 * u.uS
dt = 0.025 * u.ms
duration = 1.0 * u.ms

pre = build_pre_pc(size=n_pre)
post = build_post_pc(size=n_post)
initialize_to_steady_state(pre, post)

net = braincell.Network(name="pc_population_demo")
net.add_population("PC_pre", pre)
net.add_population("PC_post", post)
edges = net.add_edges(
    name="PC_to_PC",
    pre="PC_pre",
    post="PC_post",
    method=braincell.network.probability(
        p=connection_probability,
        seed=connection_seed,
    ),
)
pre_index = edges.pre_index
post_index = edges.post_index
weights = np.full(edges.n_edge, synaptic_weight.to_decimal(u.uS), dtype=float) * u.uS

net.add_projection(
    name="PC_to_PC_exp",
    edges="PC_to_PC",
    synapse_pool="exp",
    weight=weights,
    delay=0.0 * u.ms,
)

result = net.run(dt=dt, duration=duration)
result

In [ ]:
def trace_2d(value, unit):
    arr = np.asarray(value.to_decimal(unit), dtype=float)
    return arr.reshape((arr.shape[0], -1))


def array_2d(value):
    arr = np.asarray(value, dtype=float)
    return arr.reshape((arr.shape[0], -1))


def first_crossing_step(cross):
    steps = np.flatnonzero(np.any(cross, axis=1))
    return None if steps.size == 0 else int(steps[0] + 1)


time_ms = np.asarray(result.time.to_decimal(u.ms), dtype=float)
pre_v_mV = trace_2d(result.traces["PC_pre"]["v"], u.mV)
post_v_mV = trace_2d(result.traces["PC_post"]["v"], u.mV)
post_g_uS = trace_2d(result.traces["PC_post"]["g_syn"], u.uS)
post_i_nA = trace_2d(result.traces["PC_post"]["i_syn"], u.nA)
pre_spike = array_2d(result.spikes["PC_pre"])

pre_cross = (pre_v_mV[:-1] < 0.0) & (pre_v_mV[1:] >= 0.0)
post_cross = (post_v_mV[:-1] < 0.0) & (post_v_mV[1:] >= 0.0)
first_pre_step = first_crossing_step(pre_cross)
first_post_step = first_crossing_step(post_cross)

print("time shape:", result.time.shape)
print("PC_pre.v shape:", result.traces["PC_pre"]["v"].shape)
print("PC_post.v shape:", result.traces["PC_post"]["v"].shape)
print("PC_post.g_syn shape:", result.traces["PC_post"]["g_syn"].shape)
print("PC_pre spike shape:", result.spikes["PC_pre"].shape)
print("connection probability:", connection_probability, "seed:", connection_seed)
print("connection count:", edges.n_edge)
print("edge pairs:", list(zip(pre_index.tolist(), post_index.tolist())))
if first_pre_step is None:
    print("first pre voltage crossing: none")
else:
    print("first pre voltage crossing step:", first_pre_step, "time_ms:", time_ms[first_pre_step])
if first_post_step is None:
    print("first post voltage crossing: none")
else:
    print("first post voltage crossing step:", first_post_step, "time_ms:", time_ms[first_post_step])
print("pre crossing count by cell:", pre_cross.sum(axis=0))
print("post crossing count by cell:", post_cross.sum(axis=0))
print("postsynaptic conductance ever positive:", np.any(post_g_uS > 0.0, axis=0))
print("network spike buffer count:", pre_spike.sum(axis=0).astype(int))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
pre_colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(n_pre, 1)))
post_colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(n_post, 1)))

for idx in range(n_pre):
    axes[0].plot(time_ms, pre_v_mV[:, idx], color=pre_colors[idx], linewidth=1.2, label=f"pre[{idx}]")
axes[0].set_ylabel("pre soma v (mV)")
axes[0].set_title("Presynaptic PC population with staggered soma current clamps")

for idx in range(n_post):
    axes[1].plot(time_ms, post_v_mV[:, idx], color=post_colors[idx], linewidth=1.2, label=f"post[{idx}]")
axes[1].set_ylabel("post soma v (mV)")
axes[1].set_xlabel("time (ms)")
axes[1].set_title("Postsynaptic PC population under probabilistic PC input")

for axis, n_label in zip(axes, (n_pre, n_post)):
    axis.grid(True, alpha=0.3)
    if n_label <= 20:
        axis.legend(loc="upper right", ncol=min(4, n_label), fontsize=8)

fig.suptitle("PC population network: probabilistic PC -> PC connectivity")
fig.tight_layout()
plt.show()